<a href="https://colab.research.google.com/github/nipuninduwaraedu/SE4050-Assignment-/blob/main/SE4050_IT23414778_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import Libraries

import os
import time
import json
import zipfile
import urllib.request

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("NumPy version:", np.__version__)
print("TensorFlow version:", tf.__version__)
print("Libraries loaded successfully.")

NumPy version: 2.1.3
TensorFlow version: 2.20.0
Libraries loaded successfully.


In [ ]:
# Download dataset
DATASET_URL = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human+activity+recognition+using+smartphones.zip"
)

ZIP_PATH = "UCI_HAR_Dataset.zip"
EXTRACT_PATH = "UCI_HAR_Dataset_extracted"

print("Downloading dataset...")
urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
print("Download completed.")

print("Extracting dataset...")
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

DATASET_PATH = os.path.join(EXTRACT_PATH, "UCI HAR Dataset")

print("Dataset extracted.")
print("Dataset location:", DATASET_PATH)


Download completed.
Extracting dataset...
Dataset extracted.
Dataset location: UCI_HAR_Dataset_extracted/UCI HAR Dataset


In [ ]:
# check dataset files
print("Main dataset files:")
print("--------------------------------")

for item in os.listdir(DATASET_PATH):
    print(item)

print("\nTraining directory:")
print("--------------------------------")
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
for item in os.listdir(TRAIN_PATH):
    print(item)

print("\nTesting directory:")
print("--------------------------------")
TEST_PATH = os.path.join(DATASET_PATH, "test")
for item in os.listdir(TEST_PATH):
    print(item)


Main dataset files:
--------------------------------
test
activity_labels.txt
.DS_Store
features.txt
features_info.txt
train
README.txt

Training directory:
--------------------------------
y_train.txt
Inertial Signals
subject_train.txt
X_train.txt

Testing directory:
--------------------------------
subject_test.txt
Inertial Signals
y_test.txt
X_test.txt


In [ ]:
import os

# Let's see what folders actually exist in your workspace
print("Current workspace folders:", os.listdir("."))

# If the extraction folder exists, let's look inside it
if os.path.exists("UCI_HAR_Dataset_extracted"):
    print("Inside 'UCI_HAR_Dataset_extracted':", os.listdir("UCI_HAR_Dataset_extracted"))


Current workspace folders: ['.config', 'D:\\UCI_HAR_Dataset.zip', 'UCI_HAR_Dataset_extracted', 'D:\\UCI_HAR_Dataset', 'UCI_HAR_Dataset.zip', 'sample_data']
Inside 'UCI_HAR_Dataset_extracted': ['UCI HAR Dataset.names', 'UCI HAR Dataset.zip']


In [ ]:
import os
import zipfile

# 1. Define paths to the hidden inner zip file
inner_zip = "UCI_HAR_Dataset_extracted/UCI HAR Dataset.zip"
extract_target = "UCI_HAR_Dataset_extracted"

# 2. Unzip the inner file
print("Extracting nested dataset...")
with zipfile.ZipFile(inner_zip, "r") as zip_ref:
    zip_ref.extractall(extract_target)
print("Nested extraction finished successfully!")

# 3. Establish the correct path variable for the rest of your project
DATASET_PATH = os.path.join(extract_target, "UCI HAR Dataset")
print("Dataset path is now set to:", DATASET_PATH)


Extracting nested dataset...
Nested extraction finished successfully!
Dataset path is now set to: UCI_HAR_Dataset_extracted/UCI HAR Dataset


In [ ]:
# load activity lable

ACTIVITY_FILE = os.path.join(
    DATASET_PATH,
    "activity_labels.txt"
)

activity_df = pd.read_csv(
    ACTIVITY_FILE,
    sep=r"\s+",
    header=None,
    names=[
        "activity_id",
        "activity_name"
    ]
)

print(activity_df)

   activity_id       activity_name
0            1             WALKING
1            2    WALKING_UPSTAIRS
2            3  WALKING_DOWNSTAIRS
3            4             SITTING
4            5            STANDING
5            6              LAYING


In [ ]:
# difine sensor channel

SIGNAL_NAMES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",

    "total_acc_x",
    "total_acc_y",
    "total_acc_z",

    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z"
]

print("Number of sensor channels:", len(SIGNAL_NAMES))

for i, signal in enumerate(
    SIGNAL_NAMES,
    start=1
):
    print(i, signal)

Number of sensor channels: 9
1 body_acc_x
2 body_acc_y
3 body_acc_z
4 total_acc_x
5 total_acc_y
6 total_acc_z
7 body_gyro_x
8 body_gyro_y
9 body_gyro_z


In [18]:
#Load sensor data

def load_sensor_data(split):

    signal_folder = os.path.join(
        DATASET_PATH,
        split,
        "Inertial Signals"
    )

    channels = []

    for signal_name in SIGNAL_NAMES:

        file_path = os.path.join(
            signal_folder,
            f"{signal_name}_{split}.txt"
        )

        signal_data = pd.read_csv(
            file_path,
            sep=r"\s+",
            header=None
        ).to_numpy(
            dtype=np.float32
        )

        channels.append(
            signal_data
        )

    # Shape:
    # samples x 128 time steps x 9 channels
    X = np.stack(
        channels,
        axis=-1
    )

    return X

In [19]:
# load train and test sensor data

X_train_original = load_sensor_data(
    "train"
)

X_test_original = load_sensor_data(
    "test"
)

print(
    "Original training shape:",
    X_train_original.shape
)

print(
    "Original testing shape:",
    X_test_original.shape
)

Original training shape: (7352, 128, 9)
Original testing shape: (2947, 128, 9)


In [20]:
#load labels

y_train_original = pd.read_csv(
    os.path.join(
        DATASET_PATH,
        "train",
        "y_train.txt"
    ),
    header=None
).to_numpy().ravel()

y_test_original = pd.read_csv(
    os.path.join(
        DATASET_PATH,
        "test",
        "y_test.txt"
    ),
    header=None
).to_numpy().ravel()

print(
    "Training labels:",
    y_train_original.shape
)

print(
    "Testing labels:",
    y_test_original.shape
)

print(
    "Training label values:",
    np.unique(y_train_original)
)

print(
    "Testing label values:",
    np.unique(y_test_original)
)

Training labels: (7352,)
Testing labels: (2947,)
Training label values: [1 2 3 4 5 6]
Testing label values: [1 2 3 4 5 6]
